# Rough Heston paper plots

Recreate the calibrated and non-calibrated rough Heston figures from run 076 of the completed piecewise-constant and Legendre studies. The three shortest calibrated maturities are saved as separate smile PDFs.


In [ ]:
from pathlib import Path
import sys
import pickle

# Works when opened from the dataset folder or from the repository root.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "wiener_chaos/experiments/reporting.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from within the Wiener chaos pricing repository.")
sys.path.insert(0, str(ROOT))
from wiener_chaos.experiments.reporting import (saved_run, save_surface_pair,
                           save_combined_smiles, save_single_smile,
                           set_show_figures)

R = ROOT / "Rough Heston"
OUTPUT = R / "Plots"
RUN_ID = 76
PIECEWISE_STUDY = R / "Piecewise constant/Statistics calibration/study"
LEGENDRE_STUDY = R / "Legendre/Statistics calibration/study"
with (R / "rough_heston_data.pkl").open("rb") as f:
    data = pickle.load(f)
piecewise = saved_run(PIECEWISE_STUDY, RUN_ID)
legendre = saved_run(LEGENDRE_STUDY, RUN_ID)
S0 = data["S0"]
set_show_figures(True)  # Show each plot in the notebook and save its PDF.
print(f"Rough Heston run {RUN_ID:03d}; output: {OUTPUT}")

In [ ]:
for section, maturity_key, strike_key, market_key, model_key in [
    ("Calibrated", "maturities_calibr", "strikes_calibr", "iv_calibr", "iv_model"),
    ("Non-calibrated", "maturities_test", "strikes_test", "iv_test", "iv_model_test"),
]:
    folder = OUTPUT / section
    maturities, strikes, market = (data[maturity_key], data[strike_key], data[market_key])
    models = [piecewise[model_key], legendre[model_key]]
    labels = ["Piecewise constant", "Legendre"]
    files, mae_bp = save_surface_pair(maturities, strikes, market, models, labels, S0, folder)
    files.append(save_combined_smiles(maturities, strikes, market, models, labels, S0, folder))
    print(section, "MAE (bp):", {k: round(v, 2) for k, v in mae_bp.items()})
    for path in files:
        print("  saved", path.relative_to(ROOT))

## Short-maturity calibrated smiles

The paper shows the first three maturities individually. The filenames retain all six decimal places used by the LaTeX project.

In [ ]:
for maturity, strikes, market_iv, model_iv, legendre_iv in zip(
        data["maturities_calibr"][:3], data["strikes_calibr"][:3],
        data["iv_calibr"][:3], piecewise["iv_model"][:3],
        legendre["iv_model"][:3]):
    path = save_single_smile(maturity, strikes, market_iv, model_iv, S0,
                             OUTPUT / "Calibrated" / f"iv_smile_T_{maturity:.6f}.pdf",
                             additional_model_iv=legendre_iv)
    print("saved", path.relative_to(ROOT))